In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set(style="ticks", context="paper")

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.size'] = 16
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 13
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False 

# ===(Node - N)===
x_labels_node = ['3', '4']
node_all_counts = [4, 15]       
node_is_counts = [2, 4]          
node_ret_is = [94.90, 99.35] 

# ===(Edge - M)===
x_labels_edge = ['3', '4', '5']
edge_all_counts = [2, 12, 68]       
edge_is_counts = [1, 4, 6]         
edge_ret_is = [75.05, 95.44, 99.70] 

GLOBAL_MAX_COUNT = max(max(node_all_counts), max(edge_all_counts)) * 1.35 
GLOBAL_MIN_RET = max(0, min(min(node_ret_is), min(edge_ret_is)) - 5)  

def plot_subplot(ax, x_labels, all_counts, is_counts, ret_is, 
                 color_bar_is, color_line_is, title, marker_is,
                 show_left_axis=True, show_right_axis=True):
    
    x_p = np.arange(len(x_labels))
    width = 0.35  

    bars_all = ax.bar(x_p - width/2, all_counts, width, label='Original Features', 
                      color='lightgrey', edgecolor='grey', alpha=0.8)
    bars_is = ax.bar(x_p + width/2, is_counts, width, label='IS Features', 
                     color=color_bar_is, edgecolor='grey', alpha=0.9)
                     
    ax.set_xlabel('Order (Up to)', fontweight='bold')
    ax.set_title(title, fontweight='bold', pad=15, fontsize=18)

    ax.set_ylim(0, GLOBAL_MAX_COUNT) 

    if show_left_axis:
        ax.set_ylabel('Number of Features', fontweight='bold')
    else:
        ax.set_ylabel('')
        ax.set_yticklabels([])
        ax.tick_params(axis='y', left=False)

    ax_twin = ax.twinx()
    
    line_is = ax_twin.plot(x_p, ret_is, color=color_line_is, marker=marker_is, 
                           linestyle='-', linewidth=4, markersize=10, label='IS Features Retention')

    ax_twin.set_ylim(GLOBAL_MIN_RET, 105)

    if show_right_axis:
        ax_twin.set_ylabel('Information Retention Rate (%)', fontweight='bold', color=color_line_is)
        ax_twin.tick_params(axis='y', labelcolor=color_line_is)
    else:
        ax_twin.set_ylabel('')
        ax_twin.set_yticklabels([])
        ax_twin.tick_params(axis='y', right=False)

    ax.set_xticks(x_p)
    ax.set_xticklabels(x_labels)
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    for bar in bars_all:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + GLOBAL_MAX_COUNT*0.02, f'{height}', 
                ha='center', va='bottom', fontsize=13, fontweight='bold', color='dimgrey')

    for bar in bars_is:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + GLOBAL_MAX_COUNT*0.02, f'{height}', 
                ha='center', va='bottom', fontsize=14, fontweight='bold', color='black')

    for i, v in enumerate(ret_is):
        ax_twin.text(i, v + 1.5, f'{v:.2f}%', ha='center', color=color_line_is, 
                     fontweight='bold', fontsize=14)

    lines_1, labels_1 = ax.get_legend_handles_labels()
    lines_2, labels_2 = ax_twin.get_legend_handles_labels()
    ax.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', framealpha=0.9)

    return ax_twin

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)

plot_subplot(axes[0], x_labels_node, node_all_counts, node_is_counts, node_ret_is, 
             color_bar_is='#B2DF8A', color_line_is='#33A02C',
             title='(a) Grid Network: Node Orbits (N)', marker_is='^',
             show_left_axis=True, show_right_axis=False)

plot_subplot(axes[1], x_labels_edge, edge_all_counts, edge_is_counts, edge_ret_is, 
             color_bar_is='#A6CEE3', color_line_is='#1F78B4',
             title='(b) Grid Network: Edge Orbits (M)', marker_is='o',
             show_left_axis=False, show_right_axis=True)

plt.savefig(r"D:\Orbit_degree_LP\jia\SI\Grid_Network_Analysis_1x2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso  
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score
import re
import warnings

warnings.filterwarnings('ignore')
sns.set(style="ticks", context="paper")

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['font.size'] = 18
plt.rcParams['axes.labelsize'] = 20 
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 16 
plt.rcParams['axes.titlesize'] = 20 
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['axes.unicode_minus'] = False 

DATA_PATH = r"D:\Orbit_degree_LP\jia\all_3_6\n_72_features.csv" 

def natural_key(string_):
    return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', string_)]

def calculate_completeness_data(file_path):
    print("正在计算数据 (Ridge vs Interaction vs BlackBox)...")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"错误: 无法读取文件 {file_path}，将生成模拟数据用于演示布局。")

    all_features = [c for c in df.columns if c.startswith('M')]
    all_features.sort(key=natural_key)
    
    r2_ridge = []
    r2_interaction = []
    r2_blackbox = []
    
    targets = all_features[1:] 
    basis_cols = ['M1']
    scaler = StandardScaler()

    process_targets = targets if len(targets) < 100 else targets[0:50] 
    
    for idx, target in enumerate(process_targets):
        y = df[target].values
        basis_data = df[basis_cols].values
        X_lin = scaler.fit_transform(basis_data)

        reg_ridge = Ridge(alpha=1.0).fit(X_lin, y)
        r2_ridge.append(max(0, r2_score(y, reg_ridge.predict(X_lin))))

        limit_basis = basis_data[:, :20] if basis_data.shape[1] > 20 else basis_data
        poly = PolynomialFeatures(degree=4, include_bias=False)
        X_poly = scaler.fit_transform(poly.fit_transform(limit_basis))
        reg_poly = Lasso(alpha=0.0001, max_iter=1000).fit(X_poly, y)
        score_poly = max(0, r2_score(y, reg_poly.predict(X_poly)))
        r2_interaction.append(score_poly)

        rf = RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42).fit(X_lin, y)
        r2_blackbox.append(max(0, r2_score(y, rf.predict(X_lin))))
        
        if score_poly < 0.99:
            basis_cols.append(target)

    sorted_indices = np.argsort(r2_interaction)
    return np.array(r2_ridge)[sorted_indices], \
           np.array(r2_interaction)[sorted_indices], \
           np.array(r2_blackbox)[sorted_indices]

y_ridge, y_int, y_rf = calculate_completeness_data(DATA_PATH)
x_axis_p1 = range(len(y_int))

fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)

color_rf = '#1f77b4'    
color_ours = '#d62728'   
color_base = 'gray'    

ax.plot(x_axis_p1, y_rf, color=color_rf, linestyle=':', linewidth=3, label='Upper Bound (RF)', alpha=0.9)
ax.plot(x_axis_p1, y_int, color=color_ours, linewidth=3.5, label='Ours (Interaction)')

ax.plot(x_axis_p1, y_ridge, color=color_base, linestyle='--', linewidth=2, label='Baseline (Ridge)', alpha=0.6)

ax.fill_between(x_axis_p1, y_ridge, y_int, color='yellow', alpha=0.2, label='Captured Nonlinearity')

ax.set_xlabel('Features')
ax.set_ylabel('Explained Variance ($R^2$)')
ax.set_title('Method Completeness Proof', fontweight='bold', pad=15)
ax.legend(loc='lower right', fontsize=14)
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_ylim(-0.1, 1.1)

plt.savefig("D:\Orbit_degree_LP\jia\SI\Completeness_Proof_TNR18_Ridge.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import os

BASE_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features"
OUTPUT_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features\ISN49"

if not os.path.isdir(BASE_DIR):
    raise ValueError(f"目录不存在：{BASE_DIR}")

for i in range(0, 550):
    INPUT_PATH = os.path.join(BASE_DIR, f"n_{i}_node_features.csv")
    OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"n_{i}_features_dedup_N.csv")

    print(f"\n====== 处理文件：n_{i}_features.csv ======")

    if not os.path.isfile(INPUT_PATH):
        print(f"文件不存在，跳过：{INPUT_PATH}")
        continue

    df = pd.read_csv(INPUT_PATH)

    m_cols = [c for c in df.columns if c.startswith('N')]
    print(f"  共 {len(m_cols)} 个以 N 开头的特征列。")

    if len(m_cols) == 0:
        df_new = df.copy()
    else:
        df_m = df[m_cols]
        df_m_dedup = df_m.T.drop_duplicates(keep='first').T

        m_cols_kept = df_m_dedup.columns.tolist()
        print(f"  去重后保留 {len(m_cols_kept)} 个 N 列。")

        removed_m_cols = sorted(set(m_cols) - set(m_cols_kept))
        print(f"  被删除的重复 N 列数：{len(removed_m_cols)}")
        if removed_m_cols:
            print("  被删除的列示例：", removed_m_cols[:10])

        other_cols = [c for c in df.columns if not c.startswith('N')]
        df_new = pd.concat([df[other_cols], df_m_dedup], axis=1)

    zero_cols = [c for c in df_new.columns if c != 'label' and (df_new[c] == 0).all()]
    print(f"  检测到 {len(zero_cols)} 个全为 0 的列。")
    if zero_cols:
        print("  全 0 列示例：", zero_cols[:10])
        df_new = df_new.drop(columns=zero_cols)

    cols = df_new.columns.tolist()
    df_new = df_new[cols]

    df_new.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f"  已保存到：{OUTPUT_PATH}")
print("\n=== 全部处理完成 ===")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score
import warnings
import os

warnings.filterwarnings('ignore')

def select_and_save_is_features(file_path, output_path):
    print(f"\n====== 正在处理数据: {os.path.basename(file_path)} ======")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"  错误: 无法读取文件. {e}")
        return

    all_m_cols = [c for c in df.columns if str(c).startswith('N')]

    if not all_m_cols:
        print("  警告: 没有找到以 N 开头的特征列，跳过。")
        return

    def natural_key(string_):
        import re
        return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', string_)]
    
    all_m_cols.sort(key=natural_key)

    if 'N1' in all_m_cols:
        current_basis_cols = ['N1']
    else:
        current_basis_cols = [all_m_cols[0]]
        
    target_cols = [c for c in all_m_cols if c not in current_basis_cols]
    
    scaler = StandardScaler()
    kept_list = list(current_basis_cols)
    dropped_list = []

    for target in target_cols:
        y = df[target].values
        
        if len(np.unique(y)) <= 1:
            dropped_list.append(target)
            continue
            
        y_scaled = scaler.fit_transform(y.reshape(-1, 1)).ravel()

        basis_data = df[kept_list].values

        poly = PolynomialFeatures(degree=2, include_bias=False)
        X_poly = poly.fit_transform(basis_data)
        X_scaled = scaler.fit_transform(X_poly)

        model = Ridge(alpha=0.01) 
        model.fit(X_scaled, y_scaled)
        y_pred = model.predict(X_scaled)
        r2 = r2_score(y_scaled, y_pred)
 
        is_redundant = r2 > 0.995  
        
        if is_redundant:
            dropped_list.append(target)
        else:
            kept_list.append(target)

    print(f"  筛选完成: 原始M特征数: {len(all_m_cols)} | 保留特征: {len(kept_list)} | 剔除特征: {len(dropped_list)}")

    out_df = pd.DataFrame(kept_list, columns=['Selected_Features'])
    out_df.to_csv(output_path, index=False)
    print(f"  特征列表已保存至: {output_path}")
    
    return kept_list

if __name__ == "__main__":
    INPUT_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features"

    OUTPUT_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features\ISN995\selected_N_features"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for i in range(0, 550):
        input_file = os.path.join(INPUT_DIR, f"n_{i}_node_features.csv")
        output_file = os.path.join(OUTPUT_DIR, f"n_{i}_selected_features.csv")

        if not os.path.isfile(input_file):
            print(f"文件不存在，跳过: {input_file}")
            continue
            
        select_and_save_is_features(input_file, output_file)
        
    print("\n=== 全部数据集特征筛选完成 ===")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
import warnings
import os
import re

warnings.filterwarnings('ignore')

def calculate_reconstruction_loss(X_sub, X_full):

    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_sub_poly = poly.fit_transform(X_sub)
    
    scaler = StandardScaler()
    X_sub_scaled = scaler.fit_transform(X_sub_poly)
    
    reg = Ridge(alpha=0.01)
    reg.fit(X_sub_scaled, X_full)
    X_full_pred = reg.predict(X_sub_scaled)
    
    error_norm = np.linalg.norm(X_full - X_full_pred, ord='fro')
    base_norm = np.linalg.norm(X_full, ord='fro')
    
    if base_norm == 0:
        return 0.0
    return error_norm / base_norm

def natural_key(string_):
    return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', str(string_))]

def find_and_save_effective_boundary(data_path, is_feature_path, output_path, sample_size=2000, loss_threshold=0.05):
    print(f"\n====== 正在处理寻找有效边界: {os.path.basename(data_path)} ======")
    try:
        df = pd.read_csv(data_path)
        is_df = pd.read_csv(is_feature_path)
    except Exception as e:
        print(f"  错误: 无法读取文件. {e}")
        return

    all_m_cols = [c for c in df.columns if str(c).startswith('N')]
    if not all_m_cols:
        print("  警告: 原始数据中没有找到以 M 开头的特征列，跳过。")
        return
    all_m_cols.sort(key=natural_key)

    candidate_features = is_df.iloc[:, 0].tolist()
    candidate_features = [f for f in candidate_features if f in df.columns]
    
    if not candidate_features:
        print("  警告: 没有找到有效的候选特征，跳过。")
        return

    if len(df) > sample_size:
        df_sample = df.sample(n=sample_size, random_state=42)
    else:
        df_sample = df

    scaler_full = StandardScaler()
    X_full = scaler_full.fit_transform(df_sample[all_m_cols].values)

    current_features = []
    final_loss = 1.0
    
    for step, best_feat in enumerate(candidate_features):
        current_features.append(best_feat)

        X_sub = df_sample[current_features].values

        loss = calculate_reconstruction_loss(X_sub, X_full)
        
        print(f"  Step {step+1:02d} | 加入特征: {best_feat:<5} | 当前 Loss: {loss*100:.2f}%")

        if loss <= loss_threshold:
            final_loss = loss
            print(f" 触发截断! 在第 {step+1} 步达到 95% 信息覆盖 (Loss: {loss*100:.2f}%)")
            break

    if final_loss > loss_threshold:
        print(f"  警告: 已加入全部候选特征，但 Loss ({final_loss*100:.2f}%) 仍未降至 {loss_threshold*100}% 以下。")

    # 6. 保存最终截断的特征列表
    print(f"  总计保留核心特征数: {len(current_features)} / 原始M特征数: {len(all_m_cols)}")
    out_df = pd.DataFrame(current_features, columns=['Effective_Boundary_Features'])
    out_df.to_csv(output_path, index=False)
    print(f"  95%边界特征已保存至: {output_path}")
    
    return current_features

if __name__ == "__main__":

    RAW_DATA_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features"

    IS_FEATURE_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features\ISN995\selected_N_features"

    BOUNDARY_OUTPUT_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features\ISN995\boundary_95_features"
    os.makedirs(BOUNDARY_OUTPUT_DIR, exist_ok=True)

    LOSS_THRESHOLD = 0.05 
    SAMPLE_SIZE = 2000

    for i in range(0, 550):
        raw_file = os.path.join(RAW_DATA_DIR, f"n_{i}_node_features.csv")
        is_file = os.path.join(IS_FEATURE_DIR, f"n_{i}_selected_features.csv")
        out_file = os.path.join(BOUNDARY_OUTPUT_DIR, f"n_{i}_boundary_features.csv")

        if not os.path.isfile(raw_file) or not os.path.isfile(is_file):
            continue
            
        find_and_save_effective_boundary(
            data_path=raw_file, 
            is_feature_path=is_file, 
            output_path=out_file, 
            sample_size=SAMPLE_SIZE,
            loss_threshold=LOSS_THRESHOLD
        )
        
    print("\n=== 全部数据集 95% 拓扑边界特征截断完成 ===")

In [ ]:
import os
import pandas as pd
import re

def classify_networks_by_max_motif():
    INPUT_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features\ISN995\boundary_95_features"
    OUTPUT_FILE = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features\ISN995\network_max_motif_classification.csv"
    
    if not os.path.exists(INPUT_DIR):
        print(f"错误: 输入目录不存在 {INPUT_DIR}")
        return

    print("开始分析各个网络依赖的最高阶模体...")

    network_details = []

    category_counts = {
        "2-3 (最高N4)": 0,
        "2-4 (最高N15)": 0,
        "2-5 (最高N73)": 0,
        "2-6 (最高N480)": 0
    }

    for i in range(0, 550):
        file_path = os.path.join(INPUT_DIR, f"n_{i}_boundary_features.csv")
        
        if not os.path.isfile(file_path):
            continue
            
        df = pd.read_csv(file_path)
        
        if 'Effective_Boundary_Features' not in df.columns:
            continue
            
        features = df['Effective_Boundary_Features'].tolist()

        indices = []
        for f in features:
            match = re.search(r'\d+', str(f))
            if match:
                indices.append(int(match.group()))

        if not indices:
            continue

        max_idx = max(indices)
        
        if max_idx <= 4:
            category = "2-3 (最高N4)"
        elif max_idx <= 15:
            category = "2-4 (最高N15)"
        elif max_idx <= 73:
            category = "2-5 (最高N73)"
        else:
            category = "2-6 (最高N480)"
            
        category_counts[category] += 1
        
        network_details.append({
            "Network": f"n_{i}",
            "Max_Feature": f"M{max_idx}",
            "Category": category
        })

    if not network_details:
        print("未找到任何有效数据进行统计。")
        return
        
    details_df = pd.DataFrame(network_details)
    details_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

    total_networks = len(network_details)
    
    print("\n" + "="*50)
    print(f"统计完成！共成功分析了 {total_networks} 个网络。")
    print(f"各网络详细分类已保存至: {OUTPUT_FILE}\n")
    print("--- 整个数据集的网络最高阶分布 ---")
    
    for cat, count in category_counts.items():
        prop = (count / total_networks) * 100
        print(f"分类区间 {cat:<15}: {count:>3} 个网络, 占比 {prop:>5.2f}%")
    print("="*50)

if __name__ == "__main__":
    classify_networks_by_max_motif()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

IS_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features"
ISN_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\node_features\ISN995"
INPUT_CSV = os.path.join(ISN_DIR, "network_max_motif_classification.csv")
PNG_OUTPUT = os.path.join(ISN_DIR, "ISN995domain_comparison_original_stripe.png")
PDF_OUTPUT = os.path.join(ISN_DIR, "domain_comparison_pastel_colors.pdf")

SIZE_TITLE  = 26 
SIZE_TICKS  = 22.5 
SIZE_LEGEND = 20 
SIZE_LABEL  = 18 

ORDER_LEVELS = ["2-3", "2-4", "2-5", "2-6"]

COLORS = {
    "2-3": "#696969", 
    "2-4": "#E31A1C", 
    "2-5": "#1F78B4",
    "2-6": "#FDBF6F" 
}

CAT_MAP = {
    "2-3 (最高N4)": "2-3",
    "2-4 (最高N15)": "2-4",
    "2-5 (最高N73)": "2-5",
    "2-6 (最高N480)": "2-6"
}

HATCH_STYLE = "///"

TARGET_DOMAINS = [
    "Economic", 
    "Transportation", 
    "Technological",
    "Biological",  
    "Informational", 
    "Social"
]

def main():
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['axes.linewidth'] = 1.5 
    
    if not os.path.exists(INPUT_CSV):
        print(f"找不到文件: {INPUT_CSV}。")
        }
        valid_x_labels = TARGET_DOMAINS
    else:
        df = pd.read_csv(INPUT_CSV)
        plot_data = {}
        valid_x_labels = [] 
        
        for domain in TARGET_DOMAINS:
            sub_df = df[df['network_domain'] == domain]
            total = len(sub_df)
            domain_props = {order: 0.0 for order in ORDER_LEVELS}
            
            if total > 0:
                counts = sub_df['Category'].value_counts()
                for long_cat, count in counts.items():
                    short_cat = CAT_MAP.get(long_cat)
                    if short_cat:
                        domain_props[short_cat] = (count / total) * 100
            
            plot_data[domain] = domain_props
            valid_x_labels.append(domain)
        
    fig, ax = plt.subplots(figsize=(13, 7), dpi=300) 
    
    x = np.arange(len(valid_x_labels))
    width = 0.65  

    bottoms = np.zeros(len(valid_x_labels))

    for order in ORDER_LEVELS:
        heights = [plot_data[d][order] for d in valid_x_labels]
        c = COLORS[order]

        bars = ax.bar(x, heights, width, 
                      color=c,            
                      bottom=bottoms, 
                      edgecolor='white',      
                      hatch=HATCH_STYLE * 2,  
                      linewidth=1.5,
                      alpha=0.95,        
                      label=f'Order {order}')

        for i, bar in enumerate(bars):
            h = bar.get_height()
            if h > 4:  
                text_color = 'black' 
                
                ax.text(bar.get_x() + bar.get_width() / 2, 
                        bottoms[i] + h / 2, 
                        f'{h:.1f}%', 
                        ha='center', va='center', 
                        color=text_color,
                        fontsize=SIZE_LABEL, fontweight='bold')
        
        bottoms += heights

    ax.set_xticks(x)
    ax.set_xticklabels(valid_x_labels, fontsize=SIZE_TICKS)
    ax.set_ylabel("Percentage (%)", fontsize=SIZE_TITLE, fontweight='bold')
    ax.set_yticks(np.arange(0, 101, 20))
    ax.set_yticklabels(np.arange(0, 101, 20), fontsize=SIZE_TICKS)
    ax.set_ylim(0, 100)
    
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='-', color='#E0E0E0', alpha=0.7, linewidth=1)

    ax.legend(loc='lower center', 
              bbox_to_anchor=(0.5, 1.02),
              ncol=4,  
              frameon=False, 
              fontsize=SIZE_LEGEND, 
              columnspacing=3.0,
              handlelength=2.0,
              handleheight=1.5)

    plt.tight_layout()

    plt.savefig(PNG_OUTPUT, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score
import warnings
import os

warnings.filterwarnings('ignore')

def select_and_save_is_features(file_path, output_path):
    print(f"\n====== 正在处理数据: {os.path.basename(file_path)} ======")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"  错误: 无法读取文件. {e}")
        return

    all_m_cols = [c for c in df.columns if str(c).startswith('M')]

    if not all_m_cols:
        print("  警告: 没有找到以 M 开头的特征列，跳过。")
        return

    def natural_key(string_):
        import re
        return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', string_)]
    
    all_m_cols.sort(key=natural_key)

    if 'M1' in all_m_cols:
        current_basis_cols = ['M1']
    else:
        current_basis_cols = [all_m_cols[0]]
        
    target_cols = [c for c in all_m_cols if c not in current_basis_cols]
    
    scaler = StandardScaler()
    kept_list = list(current_basis_cols)
    dropped_list = []

    for target in target_cols:
        y = df[target].values
        
        if len(np.unique(y)) <= 1:
            dropped_list.append(target)
            continue
            
        y_scaled = scaler.fit_transform(y.reshape(-1, 1)).ravel()

        basis_data = df[kept_list].values
        poly = PolynomialFeatures(degree=2, include_bias=False)
        X_poly = poly.fit_transform(basis_data)
        X_scaled = scaler.fit_transform(X_poly)
        
        model = Ridge(alpha=0.01) 
        model.fit(X_scaled, y_scaled)
        y_pred = model.predict(X_scaled)
        r2 = r2_score(y_scaled, y_pred)
        
        is_redundant = r2 > 0.995  
        
        if is_redundant:
            dropped_list.append(target)
        else:
            kept_list.append(target)

    print(f"  筛选完成: 原始M特征数: {len(all_m_cols)} | 保留特征: {len(kept_list)} | 剔除特征: {len(dropped_list)}")
    
    out_df = pd.DataFrame(kept_list, columns=['Selected_Features'])
    out_df.to_csv(output_path, index=False)
    print(f"  特征列表已保存至: {output_path}")
    
    return kept_list

if __name__ == "__main__":
    INPUT_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313"
    
    OUTPUT_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\ISM995\selected_M_features"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    for i in range(0, 550):
        input_file = os.path.join(INPUT_DIR, f"n_{i}_features.csv")
        output_file = os.path.join(OUTPUT_DIR, f"n_{i}_selected_features.csv")
        
        if not os.path.isfile(input_file):
            print(f"文件不存在，跳过: {input_file}")
            continue
            
        select_and_save_is_features(input_file, output_file)
        
    print("\n=== 全部数据集特征筛选完成 ===")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
import warnings
import os
import re

warnings.filterwarnings('ignore')

def calculate_reconstruction_loss(X_sub, X_full):

    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_sub_poly = poly.fit_transform(X_sub)
    
    scaler = StandardScaler()
    X_sub_scaled = scaler.fit_transform(X_sub_poly)
    
    reg = Ridge(alpha=0.01)
    reg.fit(X_sub_scaled, X_full)
    X_full_pred = reg.predict(X_sub_scaled)
    
    error_norm = np.linalg.norm(X_full - X_full_pred, ord='fro')
    base_norm = np.linalg.norm(X_full, ord='fro')
    
    if base_norm == 0:
        return 0.0
    return error_norm / base_norm

def natural_key(string_):
    return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', str(string_))]

def find_and_save_effective_boundary(data_path, is_feature_path, output_path, sample_size=2000, loss_threshold=0.05):
    print(f"\n====== 正在处理寻找有效边界: {os.path.basename(data_path)} ======")
    try:
        df = pd.read_csv(data_path)
        is_df = pd.read_csv(is_feature_path)
    except Exception as e:
        print(f"  错误: 无法读取文件. {e}")
        return

    all_m_cols = [c for c in df.columns if str(c).startswith('M')]
    if not all_m_cols:
        print("  警告: 原始数据中没有找到以 M 开头的特征列，跳过。")
        return
    all_m_cols.sort(key=natural_key)

    candidate_features = is_df.iloc[:, 0].tolist()
    candidate_features = [f for f in candidate_features if f in df.columns]
    
    if not candidate_features:
        print("  警告: 没有找到有效的候选特征，跳过。")
        return

    if len(df) > sample_size:
        df_sample = df.sample(n=sample_size, random_state=42)
    else:
        df_sample = df

    scaler_full = StandardScaler()
    X_full = scaler_full.fit_transform(df_sample[all_m_cols].values)

    current_features = []
    final_loss = 1.0
    
    for step, best_feat in enumerate(candidate_features):
        current_features.append(best_feat)
        
        X_sub = df_sample[current_features].values
        
        loss = calculate_reconstruction_loss(X_sub, X_full)
        
        print(f"  Step {step+1:02d} | 加入特征: {best_feat:<5} | 当前 Loss: {loss*100:.2f}%")
        
        if loss <= loss_threshold:
            final_loss = loss
            print(f"  触发截断! 在第 {step+1} 步达到 95% 信息覆盖 (Loss: {loss*100:.2f}%)")
            break
            
    if final_loss > loss_threshold:
        print(f"  警告: 已加入全部候选特征，但 Loss ({final_loss*100:.2f}%) 仍未降至 {loss_threshold*100}% 以下。")

    print(f"  总计保留核心特征数: {len(current_features)} / 原始M特征数: {len(all_m_cols)}")
    out_df = pd.DataFrame(current_features, columns=['Effective_Boundary_Features'])
    out_df.to_csv(output_path, index=False)
    print(f"  95%边界特征已保存至: {output_path}")
    
    return current_features

if __name__ == "__main__":

    RAW_DATA_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313"
    
    IS_FEATURE_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\ISM995\selected_M_features"
    
    BOUNDARY_OUTPUT_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\ISM995\boundary_95_features"
    os.makedirs(BOUNDARY_OUTPUT_DIR, exist_ok=True)
    
    LOSS_THRESHOLD = 0.05 
    SAMPLE_SIZE = 2000 
    
    for i in range(0, 550):
        raw_file = os.path.join(RAW_DATA_DIR, f"n_{i}_features.csv")
        is_file = os.path.join(IS_FEATURE_DIR, f"n_{i}_selected_features.csv")
        out_file = os.path.join(BOUNDARY_OUTPUT_DIR, f"n_{i}_boundary_features.csv")
        
        if not os.path.isfile(raw_file) or not os.path.isfile(is_file):
            continue
            
        find_and_save_effective_boundary(
            data_path=raw_file, 
            is_feature_path=is_file, 
            output_path=out_file, 
            sample_size=SAMPLE_SIZE,
            loss_threshold=LOSS_THRESHOLD
        )
        
    print("\n=== 全部数据集 95% 拓扑边界特征截断完成 ===")

In [ ]:
import os
import pandas as pd
import re

def classify_networks_by_max_motif():
    INPUT_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\ISM995\boundary_95_features"
    OUTPUT_FILE = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\ISM995\boundary_95_network_max_motif_classification.csv"
    
    if not os.path.exists(INPUT_DIR):
        print(f"错误: 输入目录不存在 {INPUT_DIR}")
        return

    print("开始分析各个网络依赖的最高阶模体...")
    
    network_details = []
    
    category_counts = {
        "3 (最高M2)": 0,
        "3-4 (最高M12)": 0,
        "3-5 (最高M68)": 0,
        "3-6 (最高M545)": 0
    }

    for i in range(0, 550):
        file_path = os.path.join(INPUT_DIR, f"n_{i}_boundary_features.csv")
        
        if not os.path.isfile(file_path):
            continue
            
        df = pd.read_csv(file_path)
        
        if 'Effective_Boundary_Features' not in df.columns:
            continue
            
        features = df['Effective_Boundary_Features'].tolist()
        
        indices = []
        for f in features:
            match = re.search(r'\d+', str(f))
            if match:
                indices.append(int(match.group()))
                
        if not indices:
            continue
            
        max_idx = max(indices)
        
        if max_idx <= 2:
            category = "3 (最高M2)"
        elif max_idx <= 12:
            category = "3-4 (最高M12)"
        elif max_idx <= 68:
            category = "3-5 (最高M68)"
        else:
            category = "3-6 (最高M545)"
            
        category_counts[category] += 1
        
        network_details.append({
            "Network": f"n_{i}",
            "Max_Feature": f"M{max_idx}",
            "Category": category
        })

    if not network_details:
        print("未找到任何有效数据进行统计。")
        return
        
    details_df = pd.DataFrame(network_details)
    details_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    
    total_networks = len(network_details)
    
    print("\n" + "="*50)
    print(f"统计完成！共成功分析了 {total_networks} 个网络。")
    print(f"各网络详细分类已保存至: {OUTPUT_FILE}\n")
    print("--- 整个数据集的网络最高阶分布 ---")
    
    for cat, count in category_counts.items():
        prop = (count / total_networks) * 100
        print(f"分类区间 {cat:<15}: {count:>3} 个网络, 占比 {prop:>5.2f}%")
    print("="*50)

if __name__ == "__main__":
    classify_networks_by_max_motif()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

IS_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313"
ISM_DIR = r"D:\Orbit_degree_LP\jia\all_3_6\IS0313\ISM995"
INPUT_CSV = os.path.join(ISM_DIR, "boundary_95_network_max_motif_classification.csv")
PNG_OUTPUT = os.path.join(ISM_DIR, "ISM995domain_comparison_original_stripe.png")
PDF_OUTPUT = os.path.join(ISM_DIR, "domain_comparison_pastel_colors.pdf")

SIZE_TITLE  = 26  
SIZE_TICKS  = 22.5
SIZE_LEGEND = 20 
SIZE_LABEL  = 18 

ORDER_LEVELS = ["3", "3-4", "3-5", "3-6"]

COLORS = {
    "3":   "#696969",
    "3-4": "#E31A1C",
    "3-5": "#1F78B4",
    "3-6": "#FDBF6F"  
}

CAT_MAP = {
    "3 (最高M2)": "3",
    "3-4 (最高M12)": "3-4",
    "3-5 (最高M68)": "3-5",
    "3-6 (最高M545)": "3-6"
}

HATCH_STYLE = "///" 

TARGET_DOMAINS = [
    "Economic", 
    "Transportation", 
    "Technological",
    "Biological",  
    "Informational", 
    "Social"
]

def main():
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['axes.linewidth'] = 1.5 
    
    if not os.path.exists(INPUT_CSV):
        print(f"找不到文件: {INPUT_CSV}。")
        valid_x_labels = TARGET_DOMAINS
    else:
        df = pd.read_csv(INPUT_CSV)
        plot_data = {}
        valid_x_labels = [] 
        
        for domain in TARGET_DOMAINS:
            sub_df = df[df['network_domain'] == domain]
            total = len(sub_df)
            domain_props = {order: 0.0 for order in ORDER_LEVELS}
            
            if total > 0:
                counts = sub_df['Category'].value_counts()
                for long_cat, count in counts.items():
                    short_cat = CAT_MAP.get(long_cat)
                    if short_cat:
                        domain_props[short_cat] = (count / total) * 100
            
            plot_data[domain] = domain_props
            valid_x_labels.append(domain)
        
    fig, ax = plt.subplots(figsize=(13, 7), dpi=300) 
    
    x = np.arange(len(valid_x_labels))
    width = 0.65  

    bottoms = np.zeros(len(valid_x_labels))

    for order in ORDER_LEVELS:
        heights = [plot_data[d][order] for d in valid_x_labels]
        c = COLORS[order]
        
        bars = ax.bar(x, heights, width, 
                      color=c,            
                      bottom=bottoms, 
                      edgecolor='white',     
                      hatch=HATCH_STYLE * 2,  
                      linewidth=1.5,
                      alpha=0.95,        
                      label=f'Order {order}')

        for i, bar in enumerate(bars):
            h = bar.get_height()
            if h > 4:  
                text_color = 'black' 
                
                ax.text(bar.get_x() + bar.get_width() / 2, 
                        bottoms[i] + h / 2, 
                        f'{h:.1f}%', 
                        ha='center', va='center', 
                        color=text_color,
                        fontsize=SIZE_LABEL, fontweight='bold')
        
        bottoms += heights

    ax.set_xticks(x)
    ax.set_xticklabels(valid_x_labels, fontsize=SIZE_TICKS)
    ax.set_ylabel("Percentage (%)", fontsize=SIZE_TITLE, fontweight='bold')
    ax.set_yticks(np.arange(0, 101, 20))
    ax.set_yticklabels(np.arange(0, 101, 20), fontsize=SIZE_TICKS)
    ax.set_ylim(0, 100)
    
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='-', color='#E0E0E0', alpha=0.7, linewidth=1)

    ax.legend(loc='lower center', 
              bbox_to_anchor=(0.5, 1.02),
              ncol=4,  
              frameon=False, 
              fontsize=SIZE_LEGEND, 
              columnspacing=3.0,
              handlelength=2.0,
              handleheight=1.5)

    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    
    plt.tight_layout()

    plt.savefig(PNG_OUTPUT, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, KBinsDiscretizer

from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import mutual_info_score
from scipy.stats import kruskal, entropy

warnings.filterwarnings('ignore')

DIM_MAPPING = {
    'IMDB-BINARY': {'N': 14, 'M': 12},
    'NCI1':        {'N': 28, 'M': 21},
    'PTC_FM':      {'N': 18, 'M': 10},
    'MUTAG':       {'N': 11, 'M': 7},
    'PROTEINS':    {'N': 86, 'M': 80}
}

METHODS = ["Chi-Squared", "Kruskal-Wallis", "MIM", "Symmetric Uncert", "Permutation"]

def get_chi2_features(X, y, k):
    X_scaled = MinMaxScaler().fit_transform(X)
    selector = SelectKBest(chi2, k=min(k, X.shape[1]))
    selector.fit(X_scaled, y)
    return X.columns[selector.get_support()].tolist()

def get_kruskal_features(X, y, k):
    scores = []
    classes = np.unique(y)
    for col in X.columns:
        groups = [X[col][y == c].values for c in classes]
        stat, p_value = kruskal(*groups)
        scores.append(stat if not np.isnan(stat) else 0)
    top_indices = np.argsort(scores)[-min(k, X.shape[1]):][::-1]
    return X.columns[top_indices].tolist()

def get_mim_features(X, y, k):
    selector = SelectKBest(mutual_info_classif, k=min(k, X.shape[1]))
    selector.fit(X, y)
    return X.columns[selector.get_support()].tolist()

def get_su_features(X, y, k, n_bins=10):
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='uniform')
    X_binned = discretizer.fit_transform(X)
    su_scores = []
    y_counts = np.bincount(y.astype(int))
    H_y = entropy(y_counts)
    
    for i in range(X.shape[1]):
        X_col = X_binned[:, i].astype(int)
        X_counts = np.bincount(X_col)
        H_x = entropy(X_counts)
        MI = mutual_info_score(X_col, y)
        su = 0 if H_x + H_y == 0 else 2.0 * MI / (H_x + H_y)
        su_scores.append(su)
        
    top_indices = np.argsort(su_scores)[-min(k, X.shape[1]):][::-1]
    return X.columns[top_indices].tolist()

def get_permutation_features(X, y, k):
    model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    model.fit(X, y)
    result = permutation_importance(model, X, y, n_repeats=5, random_state=42, n_jobs=-1)
    top_indices = np.argsort(result.importances_mean)[-min(k, X.shape[1]):][::-1]
    return X.columns[top_indices].tolist()

def calculate_metrics(df, features, target_col, n_repeats=10):
    valid_features = [f for f in features if f in df.columns]
    # 如果没有有效特征，返回所有指标为 0
    if not valid_features: 
        return {k: 0.0 for k in ['acc_mean', 'acc_std', 'prec_mean', 'prec_std', 'rec_mean', 'rec_std', 'f1_mean', 'f1_std']}

    X = df[valid_features].values
    y = df[target_col].values
    
    metrics = {'acc': [], 'prec': [], 'rec': [], 'f1': []}

    for _ in range(n_repeats):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=np.random.randint(10000))
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
        clf.fit(X_train_scaled, y_train)
        y_pred = clf.predict(X_test_scaled)
        
        metrics['acc'].append(accuracy_score(y_test, y_pred))
        metrics['prec'].append(precision_score(y_test, y_pred, average='macro', zero_division=0))
        metrics['rec'].append(recall_score(y_test, y_pred, average='macro', zero_division=0))
        metrics['f1'].append(f1_score(y_test, y_pred, average='macro', zero_division=0))

    return {
        'acc_mean': np.mean(metrics['acc']),   'acc_std': np.std(metrics['acc']),
        'prec_mean': np.mean(metrics['prec']), 'prec_std': np.std(metrics['prec']),
        'rec_mean': np.mean(metrics['rec']),   'rec_std': np.std(metrics['rec']),
        'f1_mean': np.mean(metrics['f1']),     'f1_std': np.std(metrics['f1'])
    }

def run_comparison_for_dataset(base_dir, dataset_name, target_col):
    print(f"\n" + "="*100)
    print(f"处理数据集: {dataset_name}")
    print("="*100)
    
    orbit_data_path = os.path.join(base_dir, f"{dataset_name}_orbit_features_stats_del.csv")
    try:
        df = pd.read_csv(orbit_data_path)
    except FileNotFoundError:
        print(f"找不到数据集文件: {orbit_data_path}")
        return None

    dataset_results = {}
    
    for feat_type in ['N', 'M']:
        print(f"\n--- 正在处理 {feat_type} 特征 (目标维度 Top-K = {DIM_MAPPING[dataset_name][feat_type]}) ---")
        
        all_feats = [c for c in df.columns if c.startswith(feat_type) and c.endswith('_sum')]
        target_k = DIM_MAPPING[dataset_name][feat_type]
        
        if len(all_feats) == 0:
            print(f"警告: 没有找到 {feat_type} 特征。")
            continue
            
        X_all = df[all_feats]
        y = df[target_col].values
        
        selected_features_dict = {
            "Chi-Squared": get_chi2_features(X_all, y, target_k),
            "Kruskal-Wallis": get_kruskal_features(X_all, y, target_k),
            "MIM": get_mim_features(X_all, y, target_k),
            "Symmetric Uncert": get_su_features(X_all, y, target_k),
            "Permutation": get_permutation_features(X_all, y, target_k)
        }
        
        print(f"{'Method':<18} | {'Dim':<4} | {'Accuracy':<14} | {'Precision':<14} | {'Recall':<14} | {'F1-Score'}")
        print("-" * 95)
        
        dataset_results[feat_type] = {}
        for method_name, feats in selected_features_dict.items():
            res = calculate_metrics(df, feats, target_col)
            res['dim'] = len(feats)
            dataset_results[feat_type][method_name] = res
            
            print(f"{method_name:<18} | {res['dim']:<4} | "
                  f"{res['acc_mean']:.4f}±{res['acc_std']:.4f} | "
                  f"{res['prec_mean']:.4f}±{res['prec_std']:.4f} | "
                  f"{res['rec_mean']:.4f}±{res['rec_std']:.4f} | "
                  f"{res['f1_mean']:.4f}±{res['f1_std']:.4f}")
            
    return dataset_results

if __name__ == "__main__":
    BASE_DIR = r"D:\Orbit_degree_LP\jia\SI\TUDataset"
    DATASETS = ['IMDB-BINARY', 'NCI1', 'PTC_FM', 'MUTAG', 'PROTEINS']
    TARGET_COL = 'label'
    
    all_final_results = {}

    for ds in DATASETS:
        res = run_comparison_for_dataset(BASE_DIR, ds, TARGET_COL)
        if res:
            all_final_results[ds] = res

    print("\n\n" + "="*100)
    print("跨数据集平均性能汇总表 (Cross-Dataset Average Performance) ")
    print("="*100)

    for feat_type in ['N', 'M']:
        print(f"\n【 {feat_type} 特征平均性能 】")
        print(f"{'Method':<18} | {'Avg ACC':<14} | {'Avg PREC':<14} | {'Avg REC':<14} | {'Avg F1'}")
        print("-" * 85)
        
        for method in METHODS:
            metrics_acc, metrics_prec, metrics_rec, metrics_f1 = [], [], [], []
            
            for ds in DATASETS:
                if ds in all_final_results and feat_type in all_final_results[ds]:
                    if method in all_final_results[ds][feat_type]:
                        ds_res = all_final_results[ds][feat_type][method]
                        metrics_acc.append(ds_res['acc_mean'])
                        metrics_prec.append(ds_res['prec_mean'])
                        metrics_rec.append(ds_res['rec_mean'])
                        metrics_f1.append(ds_res['f1_mean'])
            
            if metrics_acc:
                print(f"{method:<18} | "
                      f"{np.mean(metrics_acc):.4f}±{np.std(metrics_acc):.4f} | "
                      f"{np.mean(metrics_prec):.4f}±{np.std(metrics_prec):.4f} | "
                      f"{np.mean(metrics_rec):.4f}±{np.std(metrics_rec):.4f} | "
                      f"{np.mean(metrics_f1):.4f}±{np.std(metrics_f1):.4f}")
            else:
                print(f"{method:<18} | 无数据")

    print("\n所有指标评估全部完成！")

In [ ]:
import numpy as np
import torch
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from torch_geometric.datasets import TUDataset
from torch_geometric.utils import degree
from grakel.kernels import WeisfeilerLehman, ShortestPath
import warnings

warnings.filterwarnings('ignore')

DATASETS = ['MUTAG', 'PROTEINS', 'NCI1', 'PTC_FM', 'IMDB-BINARY']

KERNELS = {
    "WL Kernel": WeisfeilerLehman(n_iter=4, normalize=True),
    "Shortest Path": ShortestPath(normalize=True)
}

def calculate_metrics_svm(K_train, K_test, y_train, y_test):
    clf = SVC(kernel='precomputed') 
    clf.fit(K_train, y_train)
    y_pred = clf.predict(K_test)
    
    return {
        'acc': accuracy_score(y_test, y_pred),
        'prec': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'rec': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='macro', zero_division=0)
    }

print("="*85)
print("开始运行图核方法基线测试 (Graph Kernels + SVM) - [PyG 数据加载版]")
print("="*85)

all_final_results = {}

for ds_name in DATASETS:
    print(f"\n加载数据集: {ds_name} ...")
    try:
        dataset = TUDataset(root='data/TUDataset', name=ds_name)
    except Exception as e:
        print(f" 加载 {ds_name} 失败: {e}")
        continue
        
    grakel_graphs = []
    labels = []
    
    print("正在转换为 GraKeL 格式...")
    for data in dataset:
        edges = set([tuple(e) for e in data.edge_index.t().tolist()])
        
        node_labels = {}
        if dataset.num_features == 0:
            deg = degree(data.edge_index[0], num_nodes=data.num_nodes, dtype=torch.long)
            for i in range(data.num_nodes):
                node_labels[i] = str(deg[i].item())
        else:
            node_labels_tensor = data.x.argmax(dim=1)
            for i in range(data.num_nodes):
                node_labels[i] = str(node_labels_tensor[i].item())
                
        grakel_graphs.append([edges, node_labels])
        labels.append(data.y.item())
        
    labels = np.array(labels)
    
    print(f"{'Method':<18} | {'ACC':<14} | {'PREC':<14} | {'REC':<14} | {'F1'}")
    print("-" * 80)
    
    all_final_results[ds_name] = {}
    
    for k_name, kernel in KERNELS.items():
        metrics_list = {'acc': [], 'prec': [], 'rec': [], 'f1': []}
        
        for _ in range(10):
            G_train, G_test, y_train, y_test = train_test_split(grakel_graphs, labels, test_size=0.3, random_state=np.random.randint(10000))
            
            try:
                K_train = kernel.fit_transform(G_train)
                K_test = kernel.transform(G_test)
                
                res = calculate_metrics_svm(K_train, K_test, y_train, y_test)
                for m in res: metrics_list[m].append(res[m])
            except Exception as e:
                print(f" {k_name} 在计算时出错: {e}")
                break
                
        if metrics_list['acc']:
            mean_acc = np.mean(metrics_list['acc'])
            mean_prec = np.mean(metrics_list['prec'])
            mean_rec = np.mean(metrics_list['rec'])
            mean_f1 = np.mean(metrics_list['f1'])
            
            all_final_results[ds_name][k_name] = {
                'acc': mean_acc, 'prec': mean_prec, 'rec': mean_rec, 'f1': mean_f1
            }
            
            print(f"{k_name:<18} | "
                  f"{mean_acc:.4f}±{np.std(metrics_list['acc']):.4f} | "
                  f"{mean_prec:.4f}±{np.std(metrics_list['prec']):.4f} | "
                  f"{mean_rec:.4f}±{np.std(metrics_list['rec']):.4f} | "
                  f"{mean_f1:.4f}±{np.std(metrics_list['f1']):.4f}")

print("\n\n" + "="*85)
print("跨数据集平均性能汇总表 (Cross-Dataset Average Performance)")
print("="*85)

print(f"{'Method':<18} | {'Avg ACC':<14} | {'Avg PREC':<14} | {'Avg REC':<14} | {'Avg F1'}")
print("-" * 80)

for k_name in KERNELS.keys():
    metrics_acc, metrics_prec, metrics_rec, metrics_f1 = [], [], [], []
    
    for ds in DATASETS:
        if ds in all_final_results and k_name in all_final_results[ds]:
            ds_res = all_final_results[ds][k_name]
            metrics_acc.append(ds_res['acc'])
            metrics_prec.append(ds_res['prec'])
            metrics_rec.append(ds_res['rec'])
            metrics_f1.append(ds_res['f1'])
    
    if metrics_acc:
        print(f"{k_name:<18} | "
              f"{np.mean(metrics_acc):.4f}±{np.std(metrics_acc):.4f} | "
              f"{np.mean(metrics_prec):.4f}±{np.std(metrics_prec):.4f} | "
              f"{np.mean(metrics_rec):.4f}±{np.std(metrics_rec):.4f} | "
              f"{np.mean(metrics_f1):.4f}±{np.std(metrics_f1):.4f}")
    else:
        print(f"{k_name:<18} | 无数据")

print("\n评估全部完成！")

In [ ]:
import os
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool, global_add_pool
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import warnings

warnings.filterwarnings('ignore')

class GCN(torch.nn.Module):
    def __init__(self, hidden_channels, num_features, num_classes, extra_feat_dim=0):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels + extra_feat_dim, num_classes)

    def forward(self, x, edge_index, batch, extra_feat=None):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index).relu()
        x = global_mean_pool(x, batch) 
        x = F.dropout(x, p=0.5, training=self.training)
        
        if extra_feat is not None and extra_feat.size(1) > 0:
            x = torch.cat([x, extra_feat], dim=-1)
            
        return self.lin(x)

class GIN(torch.nn.Module):
    def __init__(self, hidden_channels, num_features, num_classes, extra_feat_dim=0):
        super(GIN, self).__init__()
        self.conv1 = GINConv(Sequential(Linear(num_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels)))
        self.conv2 = GINConv(Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels)))
        self.conv3 = GINConv(Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels)))
        self.lin = Linear(hidden_channels + extra_feat_dim, num_classes)

    def forward(self, x, edge_index, batch, extra_feat=None):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index).relu()
        x = global_add_pool(x, batch) 
        x = F.dropout(x, p=0.5, training=self.training)
        
        if extra_feat is not None and extra_feat.size(1) > 0:
            x = torch.cat([x, extra_feat], dim=-1)
            
        return self.lin(x)

def train_and_eval(model, train_loader, test_loader, device, feat_type='None'):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()
    
    for epoch in range(50):
        model.train()
        for data in train_loader:
            data = data.to(device)
            extra = None
            if feat_type == 'ISN': extra = data.isn_feat
            elif feat_type == 'ISM': extra = data.ism_feat
            
            out = model(data.x, data.edge_index, data.batch, extra)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            extra = None
            if feat_type == 'ISN': extra = data.isn_feat
            elif feat_type == 'ISM': extra = data.ism_feat
                
            out = model(data.x, data.edge_index, data.batch, extra)
            pred = out.argmax(dim=1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(data.y.cpu().numpy())
            
    return {
        'acc': accuracy_score(all_labels, all_preds),
        'prec': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'rec': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1': f1_score(all_labels, all_preds, average='macro', zero_division=0)
    }

DATASETS = ['MUTAG', 'PROTEINS', 'NCI1', 'PTC_FM', 'IMDB-BINARY']
BASE_DIR = r"D:\Orbit_degree_LP\jia\SI\TUDataset" 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*95)
print(f" 开始运行 GNN 融合测试 (Base vs. +ISN vs. +ISM) | 设备: {device}")
print("="*95)

all_final_results = {}
MODELS_TO_RUN = ["GCN", "GCN+ISN", "GCN+ISM", "GIN", "GIN+ISN", "GIN+ISM"]

for ds_name in DATASETS:
    raw_dataset = TUDataset(root='data/TUDataset', name=ds_name)
    if raw_dataset.num_features == 0:
        import torch_geometric.transforms as T
        raw_dataset.transform = T.Constant()
        num_features = 1
    else:
        num_features = raw_dataset.num_features

    orbit_path = os.path.join(BASE_DIR, f"{ds_name}_orbit_features_stats_del.csv")
    isn_path = os.path.join(BASE_DIR, f"{ds_name}_ISN.csv")
    ism_path = os.path.join(BASE_DIR, f"{ds_name}_ISM.csv")
    
    df_orbit = pd.read_csv(orbit_path) if os.path.exists(orbit_path) else None
    
    isn_dim, ism_dim = 0, 0
    isn_matrix, ism_matrix = None, None
    
    if df_orbit is not None:
        if len(raw_dataset) != len(df_orbit):
            print(f"警告: {ds_name} 的 PyG图数量({len(raw_dataset)})与CSV行数({len(df_orbit)})不一致! 融合可能会失效。")
        else:
            if os.path.exists(isn_path):
                isn_feats = pd.read_csv(isn_path).iloc[:, 0].dropna().tolist()
                valid_isn = [f for f in isn_feats if f in df_orbit.columns]
                if valid_isn:
                    isn_dim = len(valid_isn)
                    isn_matrix = torch.tensor(StandardScaler().fit_transform(df_orbit[valid_isn].values), dtype=torch.float32)
            
            if os.path.exists(ism_path):
                ism_feats = pd.read_csv(ism_path).iloc[:, 0].dropna().tolist()
                valid_ism = [f for f in ism_feats if f in df_orbit.columns]
                if valid_ism:
                    ism_dim = len(valid_ism)
                    ism_matrix = torch.tensor(StandardScaler().fit_transform(df_orbit[valid_ism].values), dtype=torch.float32)

    dataset = []
    for i in range(len(raw_dataset)):
        data = raw_dataset[i].clone()
        data.isn_feat = isn_matrix[i].unsqueeze(0) if isn_matrix is not None else torch.empty((1, 0))
        data.ism_feat = ism_matrix[i].unsqueeze(0) if ism_matrix is not None else torch.empty((1, 0))
        dataset.append(data)
        
    print(f"\n数据集: {ds_name} (Graphs: {len(dataset)}, Classes: {raw_dataset.num_classes})")
    print(f"附加特征维度检测 => ISN: {isn_dim}, ISM: {ism_dim}")
    print(f"{'Method':<12} | {'ACC':<14} | {'PREC':<14} | {'REC':<14} | {'F1'}")
    print("-" * 80)
    
    all_final_results[ds_name] = {}
    
    model_configs = [
        ("GCN", GCN(64, num_features, raw_dataset.num_classes, extra_feat_dim=0), 'None'),
        ("GCN+ISN", GCN(64, num_features, raw_dataset.num_classes, extra_feat_dim=isn_dim), 'ISN'),
        ("GCN+ISM", GCN(64, num_features, raw_dataset.num_classes, extra_feat_dim=ism_dim), 'ISM'),
        ("GIN", GIN(64, num_features, raw_dataset.num_classes, extra_feat_dim=0), 'None'),
        ("GIN+ISN", GIN(64, num_features, raw_dataset.num_classes, extra_feat_dim=isn_dim), 'ISN'),
        ("GIN+ISM", GIN(64, num_features, raw_dataset.num_classes, extra_feat_dim=ism_dim), 'ISM')
    ]
    
    for m_name, model, feat_type in model_configs:
        model = model.to(device)
        metrics_list = {'acc': [], 'prec': [], 'rec': [], 'f1': []}
        
        for _ in range(5): 
            np.random.shuffle(dataset)
            split_idx = int(len(dataset) * 0.7)
            train_loader = DataLoader(dataset[:split_idx], batch_size=64, shuffle=True)
            test_loader = DataLoader(dataset[split_idx:], batch_size=64, shuffle=False)
            
            for layer in model.children():
                if hasattr(layer, 'reset_parameters'):
                    layer.reset_parameters()
                    
            res = train_and_eval(model, train_loader, test_loader, device, feat_type)
            for k in res: metrics_list[k].append(res[k])
            
        mean_acc, mean_prec = np.mean(metrics_list['acc']), np.mean(metrics_list['prec'])
        mean_rec, mean_f1 = np.mean(metrics_list['rec']), np.mean(metrics_list['f1'])
        
        all_final_results[ds_name][m_name] = {'acc': mean_acc, 'prec': mean_prec, 'rec': mean_rec, 'f1': mean_f1}
            
        print(f"{m_name:<12} | "
              f"{mean_acc:.4f}±{np.std(metrics_list['acc']):.4f} | "
              f"{mean_prec:.4f}±{np.std(metrics_list['prec']):.4f} | "
              f"{mean_rec:.4f}±{np.std(metrics_list['rec']):.4f} | "
              f"{mean_f1:.4f}±{np.std(metrics_list['f1']):.4f}")

print("\n\n" + "="*95)
print(" 跨数据集平均性能汇总表 (Cross-Dataset Average Performance) ")
print("="*95)

print(f"{'Method':<12} | {'Avg ACC':<14} | {'Avg PREC':<14} | {'Avg REC':<14} | {'Avg F1'}")
print("-" * 80)

for m_name in MODELS_TO_RUN:
    metrics_acc, metrics_prec, metrics_rec, metrics_f1 = [], [], [], []
    
    for ds in DATASETS:
        if ds in all_final_results and m_name in all_final_results[ds]:
            ds_res = all_final_results[ds][m_name]
            metrics_acc.append(ds_res['acc'])
            metrics_prec.append(ds_res['prec'])
            metrics_rec.append(ds_res['rec'])
            metrics_f1.append(ds_res['f1'])
    
    if metrics_acc:
        print(f"{m_name:<12} | "
              f"{np.mean(metrics_acc):.4f}±{np.std(metrics_acc):.4f} | "
              f"{np.mean(metrics_prec):.4f}±{np.std(metrics_prec):.4f} | "
              f"{np.mean(metrics_rec):.4f}±{np.std(metrics_rec):.4f} | "
              f"{np.mean(metrics_f1):.4f}±{np.std(metrics_f1):.4f}")
    else:
        print(f"{m_name:<12} | 无数据")

print("\n 融合评估全部完成！")

In [ ]:
import numpy as np
from karateclub import Graph2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from torch_geometric.datasets import TUDataset
from torch_geometric.utils import to_networkx
import warnings

warnings.filterwarnings('ignore')

DATASETS = ['MUTAG', 'PROTEINS', 'NCI1', 'PTC_FM', 'IMDB-BINARY']

print("="*85)
print("开始运行无监督图嵌入基线 (Graph2Vec + Random Forest)")
print("="*85)

all_final_results = {}

for ds_name in DATASETS:
    print(f"\n加载数据集: {ds_name} ...")
    dataset = TUDataset(root='data/TUDataset', name=ds_name)
    
    nx_graphs = []
    labels = []
    for data in dataset:
        G = to_networkx(data, to_undirected=True)
        nx_graphs.append(G)
        labels.append(data.y.item())
        
    labels = np.array(labels)
    
    print("正在训练 Graph2Vec (可能需要一会)...")
    model = Graph2Vec(wl_iterations=2, dimensions=128)
    model.fit(nx_graphs)
    X = model.get_embedding() 
    
    metrics_list = {'acc': [], 'prec': [], 'rec': [], 'f1': []}
    
    print(f"{'Method':<18} | {'ACC':<14} | {'PREC':<14} | {'REC':<14} | {'F1'}")
    print("-" * 80)
    
    for _ in range(10):
        X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3, random_state=np.random.randint(10000))
        
        clf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        metrics_list['acc'].append(accuracy_score(y_test, y_pred))
        metrics_list['prec'].append(precision_score(y_test, y_pred, average='macro', zero_division=0))
        metrics_list['rec'].append(recall_score(y_test, y_pred, average='macro', zero_division=0))
        metrics_list['f1'].append(f1_score(y_test, y_pred, average='macro', zero_division=0))
        
    mean_acc = np.mean(metrics_list['acc'])
    mean_prec = np.mean(metrics_list['prec'])
    mean_rec = np.mean(metrics_list['rec'])
    mean_f1 = np.mean(metrics_list['f1'])
    
    all_final_results[ds_name] = {
        'acc': mean_acc, 'prec': mean_prec, 'rec': mean_rec, 'f1': mean_f1
    }
    
    print(f"{'Graph2Vec':<18} | "
          f"{mean_acc:.4f}±{np.std(metrics_list['acc']):.4f} | "
          f"{mean_prec:.4f}±{np.std(metrics_list['prec']):.4f} | "
          f"{mean_rec:.4f}±{np.std(metrics_list['rec']):.4f} | "
          f"{mean_f1:.4f}±{np.std(metrics_list['f1']):.4f}")

print("\n\n" + "="*85)
print("跨数据集平均性能汇总表 (Cross-Dataset Average Performance) ")
print("="*85)

print(f"{'Method':<18} | {'Avg ACC':<14} | {'Avg PREC':<14} | {'Avg REC':<14} | {'Avg F1'}")
print("-" * 80)

metrics_acc, metrics_prec, metrics_rec, metrics_f1 = [], [], [], []

for ds in DATASETS:
    if ds in all_final_results:
        ds_res = all_final_results[ds]
        metrics_acc.append(ds_res['acc'])
        metrics_prec.append(ds_res['prec'])
        metrics_rec.append(ds_res['rec'])
        metrics_f1.append(ds_res['f1'])

if metrics_acc:
    print(f"{'Graph2Vec':<18} | "
          f"{np.mean(metrics_acc):.4f}±{np.std(metrics_acc):.4f} | "
          f"{np.mean(metrics_prec):.4f}±{np.std(metrics_prec):.4f} | "
          f"{np.mean(metrics_rec):.4f}±{np.std(metrics_rec):.4f} | "
          f"{np.mean(metrics_f1):.4f}±{np.std(metrics_f1):.4f}")
else:
    print(f"{'Graph2Vec':<18} | 无数据")

print("\n评估全部完成！")

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.datasets import Airports, WebKB
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from torch_geometric.utils import subgraph
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')

class MLP(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(MLP, self).__init__()
        self.lin1 = Linear(in_channels, hidden_channels)
        self.lin2 = Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index=None): 
        x = self.lin1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin2(x)
        return x

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=1)
        self.conv2 = GATConv(hidden_channels, out_channels, heads=1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

def get_index(feature_name):
    match = re.search(r'\d+', str(feature_name))
    return int(match.group()) if match else -1

def load_motif_features(dataset_name, base_dir):
    orbit_data_path = os.path.join(base_dir, f"{dataset_name.lower()}_orbit_features.csv") 
    isn_data_path = os.path.join(base_dir, f"{dataset_name}_ISN.csv") 
    ism_data_path = os.path.join(base_dir, f"{dataset_name}_ISM.csv") 
    
    if not os.path.exists(orbit_data_path):
        raise FileNotFoundError(f"找不到基础特征文件: {orbit_data_path}")
        
    df_orbit = pd.read_csv(orbit_data_path)
    node_ids = df_orbit['node_id'].values 
    expected_nodes = len(node_ids)
    
    df_isn = pd.read_csv(isn_data_path) if os.path.exists(isn_data_path) else None
    isn_filtered = [f for f in df_isn.iloc[:, 0].dropna().tolist() if 1 <= get_index(f) <= 480] if df_isn is not None else []
    
    df_ism = pd.read_csv(ism_data_path) if os.path.exists(ism_data_path) else None
    ism_filtered = [f for f in df_ism.iloc[:, 0].dropna().tolist() if 1 <= get_index(f) <= 545] if df_ism is not None else []
    
    scaler = StandardScaler()
    
    if len(isn_filtered) > 0 and set(isn_filtered).issubset(df_orbit.columns):
        X_isn = df_orbit[isn_filtered].values
        tensor_isn = torch.FloatTensor(scaler.fit_transform(X_isn))
    else:
        tensor_isn = torch.empty((expected_nodes, 0))
        
    if len(ism_filtered) > 0 and set(ism_filtered).issubset(df_orbit.columns):
        X_ism = df_orbit[ism_filtered].values
        tensor_ism = torch.FloatTensor(scaler.fit_transform(X_ism))
    else:
        tensor_ism = torch.empty((expected_nodes, 0))
        
    return tensor_isn, tensor_ism, node_ids

def evaluate_gnn(model_class, sub_data, x_input, num_classes, n_repeats=10):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    metrics = {'acc': [], 'prec': [], 'rec': [], 'f1': []}
    num_nodes = sub_data.num_nodes
    
    for i in range(n_repeats):
        indices = np.arange(num_nodes)
        train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=np.random.randint(10000))
        
        train_mask = torch.zeros(num_nodes, dtype=torch.bool)
        test_mask = torch.zeros(num_nodes, dtype=torch.bool)
        train_mask[train_idx] = True
        test_mask[test_idx] = True

        model = model_class(x_input.shape[1], 64, num_classes).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
        
        x = x_input.to(device)
        edge_index = sub_data.edge_index.to(device)
        y = sub_data.y.to(device)
        train_mask = train_mask.to(device)
        test_mask = test_mask.to(device)
        
        best_test_acc = 0
        best_epoch_metrics = {'acc': 0, 'prec': 0, 'rec': 0, 'f1': 0}
        
        for epoch in range(200):
            model.train()
            optimizer.zero_grad()
            out = model(x, edge_index)
            loss = F.cross_entropy(out[train_mask], y[train_mask])
            loss.backward()
            optimizer.step()
            
            model.eval()
            with torch.no_grad():
                out = model(x, edge_index)
                pred = out.argmax(dim=1)
                
                y_test_np = y[test_mask].cpu().numpy()
                pred_test_np = pred[test_mask].cpu().numpy()
                
                test_acc = accuracy_score(y_test_np, pred_test_np)
                if test_acc > best_test_acc:
                    best_test_acc = test_acc
                    best_epoch_metrics['acc'] = test_acc
                    best_epoch_metrics['prec'] = precision_score(y_test_np, pred_test_np, average='macro', zero_division=0)
                    best_epoch_metrics['rec'] = recall_score(y_test_np, pred_test_np, average='macro', zero_division=0)
                    best_epoch_metrics['f1'] = f1_score(y_test_np, pred_test_np, average='macro', zero_division=0)
                    
        for k in metrics.keys():
            metrics[k].append(best_epoch_metrics[k])
        
    return {
        'acc_mean': np.mean(metrics['acc']),   'acc_std': np.std(metrics['acc']),
        'prec_mean': np.mean(metrics['prec']), 'prec_std': np.std(metrics['prec']),
        'rec_mean': np.mean(metrics['rec']),   'rec_std': np.std(metrics['rec']),
        'f1_mean': np.mean(metrics['f1']),     'f1_std': np.std(metrics['f1'])
    }

def run_gnn_experiment(dataset_name, base_data_dir):
    print(f"\n[{dataset_name}] 正在准备 GNN 数据...")
    ds_category = 'Airports' if 'airports' in dataset_name.lower() else 'WebKB'
    
    if 'usa' in dataset_name.lower():
        ds_name_specific = 'USA'
    else:
        ds_name_specific = dataset_name.split('_')[1].capitalize()
        
    ds_path = os.path.join(base_data_dir, ds_category)
    
    dataset = Airports(root=ds_path, name=ds_name_specific) if ds_category == 'Airports' else WebKB(root=ds_path, name=ds_name_specific)
    data = dataset[0]
    num_classes = dataset.num_classes
    
    try:
        tensor_isn, tensor_ism, node_ids = load_motif_features(dataset_name, base_data_dir)
    except Exception as e:
        print(f"数据加载失败, 跳过 {dataset_name}: {e}")
        return None

    subset = torch.tensor(node_ids, dtype=torch.long)
    sub_edge_index, _ = subgraph(subset, data.edge_index, relabel_nodes=True)
    sub_y = data.y[subset]
    sub_x_base = data.x[subset] if data.x is not None else torch.ones((len(subset), 1))

    sub_data = Data(x=sub_x_base, edge_index=sub_edge_index, y=sub_y)
    sub_data.num_nodes = len(subset)

    print(f"  --> 原图节点数: {data.num_nodes} | 截取后有效子图节点数: {sub_data.num_nodes}")

    inputs = {
        "1. Base (Original Features)": sub_x_base,
        "2. Base + ISN (Node Motifs)": torch.cat([sub_x_base, tensor_isn], dim=1),
        "3. Base + ISM (Edge Motifs)": torch.cat([sub_x_base, tensor_ism], dim=1),
        "4. Base + ISN + ISM (All)": torch.cat([sub_x_base, tensor_isn, tensor_ism], dim=1)
    }

    models = {
        "MLP": MLP,
        "GCN": GCN, 
        "GraphSAGE": GraphSAGE,
        "GAT": GAT
    }
    
    results = {}

    for model_name, model_class in models.items():
        print(f"\n  [启动 {model_name} 评估]")
        print(f"    {'Feature Combination':<30} | {'Dim':<4} | {'Accuracy':<14} | {'Precision':<14} | {'Recall':<14} | {'F1-Score'}")
        print("    " + "-" * 105)
        
        for feat_name, x_input in inputs.items():
            res = evaluate_gnn(model_class, sub_data, x_input, num_classes)
            res_key = f"{model_name} | {feat_name}"
            
            results[res_key] = {
                "dim": x_input.shape[1],
                "acc": res['acc_mean'], "acc_std": res['acc_std'],
                "prec": res['prec_mean'], "prec_std": res['prec_std'],
                "rec": res['rec_mean'], "rec_std": res['rec_std'],
                "f1": res['f1_mean'], "f1_std": res['f1_std']
            }
            
            print(f"    {feat_name:<30} | {x_input.shape[1]:<4} | "
                  f"{res['acc_mean']:.4f}±{res['acc_std']:.4f} | "
                  f"{res['prec_mean']:.4f}±{res['prec_std']:.4f} | "
                  f"{res['rec_mean']:.4f}±{res['rec_std']:.4f} | "
                  f"{res['f1_mean']:.4f}±{res['f1_std']:.4f}")

    return results

if __name__ == "__main__":
    datasets = [
        'airports_brazil', 
        'airports_europe', 
        'airports_usa', 
        'webkb_texas', 
        'webkb_wisconsin'
    ] 
    
    BASE_DATA_DIR = r"D:\Orbit_degree_LP\jia\SI\Datasets"
    
    print("="*115)
    print("节点分类突破实验：基于提取子图的 5 个数据集交叉验证") 
    print("="*115)

    all_results = {}
    valid_dataset_count = 0

    for ds in datasets:
        res = run_gnn_experiment(ds, BASE_DATA_DIR)
        if res is not None:
            valid_dataset_count += 1
            for key, val in res.items():
                if key not in all_results:
                    all_results[key] = []
                all_results[key].append(val)

    if valid_dataset_count > 0:
        print("\n\n" + "="*115)
        print(f"跨 {valid_dataset_count} 个数据集节点分类性能汇总表 (Cross-Dataset Average)") 
        print("="*115)
        print(f"{'Model & Feature Combination':<40} | {'Avg Dim':<8} | {'Avg ACC':<14} | {'Avg PREC':<14} | {'Avg REC':<14} | {'Avg F1'}")
        print("-" * 115)

        for key, res_list in all_results.items():
            if len(res_list) == 0: continue
            
            avg_dim = np.mean([x["dim"] for x in res_list])
            
            acc_vals = [x["acc"] for x in res_list]
            prec_vals = [x["prec"] for x in res_list]
            rec_vals = [x["rec"] for x in res_list]
            f1_vals = [x["f1"] for x in res_list]
            
            print(f"{key:<40} | {avg_dim:<8.1f} | "
                  f"{np.mean(acc_vals):.4f}±{np.std(acc_vals):.4f} | "
                  f"{np.mean(prec_vals):.4f}±{np.std(prec_vals):.4f} | "
                  f"{np.mean(rec_vals):.4f}±{np.std(rec_vals):.4f} | "
                  f"{np.mean(f1_vals):.4f}±{np.std(f1_vals):.4f}")
        
        print("="*115 + "\n")
    else:
        print("\n没有成功加载任何数据集，请检查 CSV 文件路径及命名！")